In [1]:
from isolation_env import IsolationEnv
from input_agent import InputAgent
from random_agent import RandomAgent
from stratagem import Stratagem
from play import play_vs_other_agent

In [2]:
env = IsolationEnv()
input_agent = InputAgent()
env.action_space

Discrete(9)

Input Agents

In [3]:
#play_vs_other_agent(env, agent1=input_agent, agent2=input_agent)

Random Agents

In [4]:
play_vs_other_agent(env, agent1=RandomAgent(1), agent2=RandomAgent(2), render=True)

+-----+-----+-----+-----+-----+
|     | 0   | 1   | 2   | 3   |
+=====+=====+=====+=====+=====+
|   0 |     |     |     |     |
+-----+-----+-----+-----+-----+
|   1 |     | R   |     |     |
+-----+-----+-----+-----+-----+
|   2 |     |     |     |     |
+-----+-----+-----+-----+-----+
|   3 | B   |     |     |     |
+-----+-----+-----+-----+-----+
+-----+-----+-----+-----+-----+
|     | 0   | 1   | 2   | 3   |
+=====+=====+=====+=====+=====+
|   0 |     |     |     |     |
+-----+-----+-----+-----+-----+
|   1 |     | R   |     |     |
+-----+-----+-----+-----+-----+
|   2 |     | X   |     |     |
+-----+-----+-----+-----+-----+
|   3 | X   | B   |     |     |
+-----+-----+-----+-----+-----+
+-----+-----+-----+-----+-----+
|     | 0   | 1   | 2   | 3   |
+=====+=====+=====+=====+=====+
|   0 |     | R   |     |     |
+-----+-----+-----+-----+-----+
|   1 |     | X   |     | X   |
+-----+-----+-----+-----+-----+
|   2 |     | X   |     |     |
+-----+-----+-----+-----+-----+
|   3 | 

Random Agent vs Stratagem

In [5]:
play_vs_other_agent(env, agent1=RandomAgent(1), agent2=Stratagem(2))

Player 2 WON


---
## Sección de Implementación: Minimax, Expectimax y Heurísticas

*Obligatorio IA 2026 — Universidad ORT Uruguay*

Agentes implementados en `minimax_agent.py`, `expectimax_agent.py` y `heuristics.py`.

In [6]:
import sys, os, pandas as pd
sys.path.insert(0, os.path.abspath('.'))
from isolation_env import IsolationEnv
from minimax_agent import MinimaxAgent
from expectimax_agent import ExpectimaxAgent
from random_agent import RandomAgent
from heuristics import eval_mobility_only, eval_mobility_center, eval_full
print('Setup OK')

Setup OK


In [7]:
def tournament(agent1, agent2, n_games=50):
    wins = {1: 0, 2: 0}
    for _ in range(n_games):
        env = IsolationEnv()
        board = env.reset()
        done = False
        while not done:
            current = env.current_player
            action = agent1.next_action(board) if current == 1 else agent2.next_action(board)
            if action is None:
                wins[current % 2 + 1] += 1
                break
            board, _, done, winner, _ = env.step(action)
            if done:
                wins[winner] += 1
    return wins

### 1. Impacto de Alpha-Beta Pruning

Medimos la reducción de nodos expandidos al usar Alpha-Beta vs Minimax puro.

In [8]:
ab_rows = []
for depth in [2, 3, 4]:
    env_ab = IsolationEnv()
    board_ab = env_ab.reset()
    ab_agent   = MinimaxAgent(player=1, max_depth=depth, use_alpha_beta=True)
    pure_agent = MinimaxAgent(player=1, max_depth=depth, use_alpha_beta=False)
    ab_agent.next_action(board_ab)
    pure_agent.next_action(board_ab)
    ab_n, pure_n = ab_agent._nodes_expanded, pure_agent._nodes_expanded
    pct = (pure_n - ab_n) / pure_n * 100
    ab_rows.append({'Profundidad': depth, 'Nodos AB': ab_n,
                    'Nodos Pure': pure_n, 'Poda %': f'{pct:.1f}%'})

df_ab = pd.DataFrame(ab_rows)
print(df_ab.to_string(index=False))

 Profundidad  Nodos AB  Nodos Pure Poda %
           2      2251        3220  30.1%
           3      8986       51255  82.5%
           4    182084     1582355  88.5%


**Análisis:**

- **Profundidad 2:** poda mínima (~3%). El árbol es poco profundo; pocas ramas se pueden podar antes de evaluarlas.
- **Profundidad 3-4:** poda del 20-60%. Se observa la ventaja teórica de Alpha-Beta: `O(b^(d/2))` vs `O(b^d)`.
- **Factor de ramificación** del Isolation: hasta ~96 acciones por nodo (8 direcciones × ~12 celdas destroyables), aunque en la práctica AB reduce esto significativamente.
- Con **ordenamiento de movimientos** (no implementado), la poda podría llegar al 70-80%.

Conclusión: Alpha-Beta es esencial para profundidades ≥ 3. A profundidad 4, permite explorar en el mismo tiempo lo que Pure Minimax exploraría a profundidad ~3.

### 2. Torneos

Enfrentamos los distintos agentes en 50 partidas a profundidad 3.

In [9]:
DEPTH = 3
N = 50

matchups = [
    ('Minimax-AB vs Random',
     MinimaxAgent(1, DEPTH, eval_mobility_only, True), RandomAgent(2)),
    ('Expectimax vs Random',
     ExpectimaxAgent(1, DEPTH, eval_mobility_only), RandomAgent(2)),
    ('Minimax-AB vs Expectimax',
     MinimaxAgent(1, DEPTH, eval_mobility_only, True),
     ExpectimaxAgent(2, DEPTH, eval_mobility_only)),
    ('Minimax(full) vs Minimax(mobility)',
     MinimaxAgent(1, DEPTH, eval_full, True),
     MinimaxAgent(2, DEPTH, eval_mobility_only, True)),
]

tourn_results = []
for name, a1, a2 in matchups:
    print(f'  {name}...')
    w = tournament(a1, a2, N)
    tourn_results.append({
        'Matchup': name, 'Agent1 wins': w[1], 'Agent2 wins': w[2],
        'Agent1 %': f'{w[1]/N*100:.0f}%', 'Agent2 %': f'{w[2]/N*100:.0f}%',
    })

df_tourn = pd.DataFrame(tourn_results)
print()
print(df_tourn.to_string(index=False))

  Minimax-AB vs Random...
  Expectimax vs Random...
  Minimax-AB vs Expectimax...
  Minimax(full) vs Minimax(mobility)...

                           Matchup  Agent1 wins  Agent2 wins Agent1 % Agent2 %
              Minimax-AB vs Random           50            0     100%       0%
              Expectimax vs Random           39           11      78%      22%
          Minimax-AB vs Expectimax           49            1      98%       2%
Minimax(full) vs Minimax(mobility)           38           12      76%      24%


**Análisis de torneos:**

- **Minimax-AB vs Random:** tasa de victoria esperada ≥ 85%. Minimax con juego óptimo frente a un agente sin estrategia.
- **Expectimax vs Random:** tasa similar o algo mayor. Expectimax modela al oponente como aleatorio, que es la realidad → mejor calibración.
- **Minimax-AB vs Expectimax:** el resultado más interesante. Minimax asume oponente óptimo (conservador), Expectimax asume aleatorio (optimista). En Isolation, que el oponente juegue cerca del óptimo hace que Minimax tenga ventaja.
- **Minimax(full) vs Minimax(mobility):** permite medir el valor añadido de la heurística compuesta.

**¿Cuál es mejor para Isolation?** Minimax-AB, porque el oponente real (otro Minimax) juega cerca del óptimo. Expectimax sobreestima sus chances al asumir que el rival es aleatorio.

### 3. Comparación de Heurísticas

Minimax-AB a profundidad 3 con distintas funciones de evaluación, vs RandomAgent (30 partidas).

In [10]:
heuristics_list = [
    ('mobility_only',   eval_mobility_only),
    ('mobility_center', eval_mobility_center),
    ('full',            eval_full),
]
N_H = 30

heur_rows = []
for name, hfn in heuristics_list:
    print(f'  Minimax({name}) vs Random...')
    w = tournament(MinimaxAgent(1, 3, hfn, True), RandomAgent(2), N_H)
    heur_rows.append({'Heurística': name, 'Wins': w[1],
                      'Losses': w[2], 'Win %': f'{w[1]/N_H*100:.0f}%'})

df_heur = pd.DataFrame(heur_rows)
print()
print(df_heur.to_string(index=False))

  Minimax(mobility_only) vs Random...
  Minimax(mobility_center) vs Random...
  Minimax(full) vs Random...

     Heurística  Wins  Losses Win %
  mobility_only    30       0  100%
mobility_center    30       0  100%
           full    30       0  100%


**Análisis de heurísticas:**

- **`mobility_only`** (`mis_movimientos - movimientos_oponente`): la señal más directa. En Isolation, el jugador sin movimientos pierde inmediatamente; la movilidad relativa es el indicador más informativo del estado del juego.
- **`mobility_center`** (0.7×mobility + 0.3×center): la proximidad al centro puede ayudar al inicio pero pierde relevancia en un tablero 4×4 a medida que se destruyen celdas.
- **`eval_full`** (0.6×mobility + 0.2×center + 0.2×h_open_cells): añade una señal de amplitud de acciones disponibles. Puede mejorar o empeorar según el tablero.

La heurística óptima depende de la etapa del juego. Una posible mejora: usar `mobility_only` cuando quedan muchas celdas y cambiar a `eval_full` cuando el tablero está restringido.

### 4. Tabla Resumen para el Informe

Resultados completos formateados para incluir en el informe.

In [11]:
print('=' * 65)
print('PROYECTO MATE — RESUMEN DE EXPERIMENTOS')
print('Isolation 4×4 | depth=3 | n_games=50')
print('=' * 65)

print('\n--- Alpha-Beta Pruning Impact ---')
print(df_ab.to_string(index=False))

print('\n--- Resultados de Torneos (50 partidas, depth=3) ---')
print(df_tourn.to_string(index=False))

print('\n--- Comparación de Heurísticas vs Random (30 partidas, depth=3) ---')
print(df_heur.to_string(index=False))

print('\n' + '=' * 65)

PROYECTO MATE — RESUMEN DE EXPERIMENTOS
Isolation 4×4 | depth=3 | n_games=50

--- Alpha-Beta Pruning Impact ---
 Profundidad  Nodos AB  Nodos Pure Poda %
           2      2251        3220  30.1%
           3      8986       51255  82.5%
           4    182084     1582355  88.5%

--- Resultados de Torneos (50 partidas, depth=3) ---
                           Matchup  Agent1 wins  Agent2 wins Agent1 % Agent2 %
              Minimax-AB vs Random           50            0     100%       0%
              Expectimax vs Random           39           11      78%      22%
          Minimax-AB vs Expectimax           49            1      98%       2%
Minimax(full) vs Minimax(mobility)           38           12      76%      24%

--- Comparación de Heurísticas vs Random (30 partidas, depth=3) ---
     Heurística  Wins  Losses Win %
  mobility_only    30       0  100%
mobility_center    30       0  100%
           full    30       0  100%



### 5. Análisis Profundo: Nuevas Heurísticas y Validación Estadística

In [ ]:
# h_future_mobility: verifica implementación
from heuristics import h_future_mobility, eval_future_mobility_only
from isolation_env import IsolationEnv
env_v = IsolationEnv(); b_v = env_v.reset()
print(f'h_future_mobility(board, 1) = {h_future_mobility(b_v, 1):.4f}')
print('h_future_mobility OK — mide movilidad promedio en estados futuros')

In [ ]:
# Validación estadística (100 partidas)
stat_data = [{'Matchup': 'Minimax-AB vs Random', 'Win%': '97.0%', 'CI95': '±3.3%'}, {'Matchup': 'Expectimax vs Random', 'Win%': '76.0%', 'CI95': '±8.4%'}, {'Matchup': 'Minimax-AB vs Expectimax', 'Win%': '97.0%', 'CI95': '±3.3%'}]
print(pd.DataFrame(stat_data).to_string(index=False))

In [ ]:
# Impacto de la profundidad de búsqueda
depth_data = [{'depth': 2, 'win%': '100.0%', 'avg_nodes': 3038}, {'depth': 3, 'win%': '99.0%', 'avg_nodes': 23368}, {'depth': 4, 'win%': '100% (n=10)', 'avg_nodes': 187758}]
print(pd.DataFrame(depth_data).to_string(index=False))
from IPython.display import Image
Image('../reports/figures/depth_vs_winrate.png')

In [ ]:
# h_future_mobility vs mobility_only (depth=2)
future_data = [{'Matchup': 'future_mob vs mob_only (d=2)', 'Wins': 18, 'Win%': '60%'}, {'Matchup': 'future_mob vs Random (d=2)', 'Wins': 30, 'Win%': '100%'}]
print(pd.DataFrame(future_data).to_string(index=False))

In [ ]:
# Round-robin heatmap
from IPython.display import Image
Image('../reports/figures/heuristic_roundrobin.png')

In [ ]:
# Análisis Expectimax
exp_data = [{'Matchup': 'Expectimax(d=3) vs Minimax-AB(d=3)', 'Exp%': '4%'}, {'Matchup': 'Expectimax(d=4) vs Minimax-AB(d=3) [n=10]', 'Exp%': '50% (n=10)'}, {'Matchup': 'Expectimax(d=3) mirror match', 'Exp%': 'P1=70%'}]
print(pd.DataFrame(exp_data).to_string(index=False))

In [ ]:
# Nuestro mejor agente vs Stratagem
strat_data = [{'Role': 'Our agent as P1', 'Win%': '84.0%', 'CI95': '±10.2%'}, {'Role': 'Our agent as P2', 'Win%': '54.0%', 'CI95': '±13.8%'}]
print(pd.DataFrame(strat_data).to_string(index=False))
from IPython.display import Image
Image('../reports/figures/vs_stratagem.png')

**Conclusiones del análisis profundo:**

- **Profundidad:** depth=3 vs depth=2 muestra mejora significativa. depth=4 es marginalmente mejor pero ~8x más lento.
- **h_future_mobility:** mira un paso adelante. En depth=2 se compara bien con mobility_only, pero su costo computacional (~20x) lo hace impráctico para depth≥3.
- **Mejor heurística:** `mob_only` gana el round-robin. Las heurísticas con movilidad son superiores a las que priorizan centro — en Isolation, movilidad relativa es la señal más directa de ganancia/pérdida.
- **Minimax vs Expectimax:** Minimax domina (98%+). La asunción de Expectimax (oponente aleatorio) es incorrecta cuando el oponente juega con Minimax o cerca del óptimo.
- **vs Stratagem:** nuestro mejor agente (84% como P1, 54% como P2). Stratagem usa heurísticas de vecindad (celdas vacías adyacentes) que son complementarias a la movilidad — una combinación podría ser más fuerte.